# PSH02 — Volatility × signed aggressive flow and perpetual-futures return prediction

Preregistered in `discussions/Predictive_State_HMM/002_chatgpt_psh02_preregistration.md`. **Historical exploratory only. No trading, no option P&L or order placement.** PSH01 results are preserved.

In [ ]:
from pathlib import Path
import os, sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p/"src"/"data_engineering").exists())
sys.path.insert(0,str(root/"src"))
from dotenv import load_dotenv
load_dotenv(root/".env")
from data_engineering.modeling import predictive_state_flow as sf
print("Module:", sf.VERSION, "root:", root)

In [ ]:
data_root = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser()
spot_dir = data_root/"history"/"price_5m"
perp_dir = Path(os.environ.get("BINANCE_UM_DIR",str(data_root.parent/"Binance"))).expanduser()
out_dir = root/"notebooks"/"Predictive_State_HMM"/"02"
cfg = sf.Spec()
print("Spot:",spot_dir,"\nPerps:",perp_dir,"\nOutput:",out_dir)
sf.registration(cfg)

## Frozen historical run

Reads local Binance 5m spot and Binance USD-M 1m perpetual Parquet. Fits PSH01 K=4 structure on TRAIN only, then compares B0/B1/B2/B3/B4/nonlinear on the exact same nonzero-return origins. Scores **5m primary** and 30m secondary. Each asset and horizon may take some time. The run writes only compact aggregate outputs under `02/`.

In [ ]:
result = sf.run(spot_dir, perp_dir, out_dir, cfg)
result["decisions"]

## Primary directional prediction

The negative comparison delta means the first named method has lower predictive log loss. `upper_familywise` is the preregistered one-sided familywise bootstrap cutoff. **Neither a nominal pass nor a favorable historical retrospective score is independent prospective confirmation.**

In [ ]:
from IPython.display import display
display(result["scores"].query("horizon_min == 5").round(6))
display(result["comparisons"].query("horizon_min == 5").round(6))

## 30-minute secondary outcome and perp cost scenarios

The cost scenarios use **perpetual close-to-close returns** and an assumed 12 bp round trip, but not executable entry fills, funding, fees specific to account tier, or margin constraints. Do not treat top-decile ranking or isolated wins as deployable signals.

In [ ]:
display(result["scores"].query("horizon_min == 30").round(6))
display(result["perp_scenarios"].round(4))

## Saved artifacts and review contract

Commit only generated aggregate `02/` outputs and this executed notebook; **do not push minute-level or per-origin source data**. PSH01, HSMM and VD19 artifacts must remain unchanged. The decision JSON is the binding recorded historical result.

In [ ]:
import json
for name in ["02_forecast_scores.csv","02_paired_day_comparisons.csv",
             "02_perp_cost_scenarios.csv","02_coverage.csv",
             "02_decision.json","02_manifest.json"]:
    path=out_dir/name
    print(name, "saved" if path.is_file() else "MISSING", path.stat().st_size if path.is_file() else "")